# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading, exploring, and processing the FAIR² ("FAIR squared") dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) accessible from the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the metadata object (do NOT subscript)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Dataset description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print("\nKeywords:")
for kw in getattr(dataset.metadata, 'keywords', []):
    print(f"- {kw}")

## 2. Data Overview
Review available **record sets**, their fields, and their `@id`s.

Below, we enumerate each record set in the dataset and the corresponding fields (columns).

In [ ]:
# List available record sets and their fields with @ids
record_sets = list(dataset.metadata.recordSet)
if record_sets:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']}")
        rs_name = rs.get('name', '')
        if rs_name:
            print(f"  Name: {rs_name}")
        # List fields
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"  Fields ({len(fields)}):")
        for f in fields:
            print(f"    - Field @id: {f['@id']} | {f.get('name','')}")
        print()
else:
    print("Warning: No record sets found in the Croissant schema. The dataset may be using a flat distribution or old schema. Attempting to enumerate distributions.")

# If there are no record sets, try distributions (files with tabular data)
distributions = getattr(dataset.metadata, 'distribution', [])
for dist in distributions:
    print(f"Distribution @id: {dist['@id']}")
    print(f"  EncodingFormat:  {dist.get('encodingFormat','')}")
    print(f"  ContentUrl: {dist.get('contentUrl','')}")

## 3. Data Extraction
Load records from a specific record set (`@id`) into a DataFrame for analysis.

If the dataset doesn't explicitly define `recordSet`, use the primary tabular data distribution instead.

In [ ]:
# Select a recordSet `@id` or fallback to a tabular distribution @id
if record_sets:
    first_record_set_id = record_sets[0]['@id']
    print(f"Using recordSet @id: {first_record_set_id}")
    record_set_ids = [rs['@id'] for rs in record_sets]
else:
    # Fallback: Use distribution
    dist_ids = [dist['@id'] for dist in getattr(dataset.metadata, 'distribution', [])]
    if dist_ids:
        print(f"Using distribution @id: {dist_ids[0]}")
        record_set_ids = [dist_ids[0]]
    else:
        raise RuntimeError("No recordSet or distribution available for extraction.")

# Load all listed record sets (or distributions) into DataFrames
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set: {rs_id} | Shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()[:8]} ...\n")
    except Exception as e:
        print(f"Failed to load {rs_id}: {e}")

# Display the first few rows from the primary record set
primary_id = record_set_ids[0]
if primary_id in dataframes:
    display_cols = dataframes[primary_id].columns.tolist()[:12]
    print(f"First record rows (columns subset):")
    display(dataframes[primary_id][display_cols].head())

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic data processing steps:

- Filtering protein records by a numeric field (e.g., `Coverage(%)`)
- Normalizing the chosen numeric field
- Optionally, grouping by a string/categorical field (e.g., `Gene Symbol` or similar)

All fields will be referenced using their `@id`, as per FAIR Croissant guidance.

In [ ]:
# Choose a numeric field @id for demonstration. Adjust to schema's actual field ids.
df = dataframes[primary_id]

# Try to auto-detect a likely numeric field (id or name) for coverage percentage
possible_numeric_fields = [c for c in df.columns if 'coverage' in c.lower() or c.lower() in ['coverage', 'coverage(%)', 'mw', 'molecular_weight', 'count', 'abundance']]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # use the first match
else:
    numeric_field = df.select_dtypes(include='number').columns[0]  # fallback: first numeric col
print(f"Using numeric field: {numeric_field}")

# Filter records (e.g., Coverage(%) > 10)
threshold = 10
filtered_df = df.copy()
filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field], errors='coerce') > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold}, count: {len(filtered_df)}")
display(filtered_df.head())

# Normalize selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field], errors='coerce') - filtered_df[numeric_field].astype(float).mean()
) / filtered_df[numeric_field].astype(float).std()
print(f"\nNormalized '{numeric_field}' head:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical field
possible_group_fields = [c for c in df.columns if 'gene' in c.lower() or 'accession' in c.lower() or 'symbol' in c.lower() or c.lower() in ['gene_symbol', 'accession']]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).agg({numeric_field: 'mean', f'{numeric_field}_normalized': 'mean'})
    print(f"\nGrouped data by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the chosen numeric field and display relationships between two main attributes if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce'), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If possible, plot numeric field vs normalized
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(5,5))
    sns.scatterplot(
        x=filtered_df[numeric_field],
        y=filtered_df[f"{numeric_field}_normalized"]
    )
    plt.xlabel(numeric_field)
    plt.ylabel(f"{numeric_field} (normalized)")
    plt.title(f"Relationship: {numeric_field} vs Normalized")
    plt.show()

# If a group field exists, visualize top values
if group_field:
    plt.figure(figsize=(8,3))
    top_groups = (
        filtered_df[group_field]
        .value_counts()
        .iloc[:10].index
    )
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df[filtered_df[group_field].isin(top_groups)], palette='tab10')
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field} (Top 10)")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have:

- Programmatically loaded the FAIR² mass spectrometry dataset using `mlcroissant`
- Explored dataset structure and extracted tabular protein records
- Applied basic filtering and normalization to a protein attribute (coverage or equivalent)
- Visualized distributions for in-depth biological or data quality inspection

Further processing (e.g., statistical testing, bioinformatics workflows, etc.) can build on these robust FAIR data foundations.

For more information see the [Croissant documentation](https://mlcommons.org/croissant/) and [FAIR² schema description](https://sen.science/doi/10.71728/senscience.vq4a-28xa).